In [8]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/thedevastator/tinystories-narrative-classification/validation.csv
/kaggle/input/datasets/thedevastator/tinystories-narrative-classification/train.csv


# Introduction

This notebook contains code for a micro-Small Language Model (< 2 Million parameters) consisting of a Language Model with 1-2 transformer blocks and multi-headed, causal self-attention. This model is pretrained on the TinyStories dataset for the next-token prediction task. The pretrained model is then repeatedly input generated text to generate a short story.

# Steps:
1. **Data Exploration**
2. **Data Preprocessing**
     1. Tokenization
     2. Saving data to memory
     3. Input-target pairing
3. **Building Model Architecture**
    1. Projection to Token Embedding Space
    2. Projection to Positional Embedding Space (for later; unclear on this)
    3. Attention
    4. Feed-forward Network/ Multi-layer Perceptrons
    5. Layer Normalization
    6. Transformer
    7. Small Language Model (contains embedding projection, transformer blocks, LM Head, etc.)
4. **Pre-training**
    1. Defining cross-entropy loss function
    2. Updating parameters using backpropagation
    3. Pre-training using training data
    4. Validate model on validation dataset
    5. Visualize and analyze training and validation losses
    6. Hyperparameter tuning and repeat training and validation
5. **Short Story Generation**
6. **Explore and Evaluate Generated Output**


# Step 1: Data Exploration

In [9]:
if not os.path.exists('./data'):
    os.makedirs('./data')
    
kagglehub.dataset_download('thedevastator/tinystories-narrative-classification',
                           path = 'train.csv', output_dir = './data')
kagglehub.dataset_download('thedevastator/tinystories-narrative-classification',
                           path = 'validation.csv', output_dir = './data')

'/kaggle/input/datasets/thedevastator/tinystories-narrative-classification/validation.csv'

In [10]:
train = pd.read_csv('/kaggle/input/datasets/thedevastator/tinystories-narrative-classification/train.csv')
val = pd.read_csv('/kaggle/input/datasets/thedevastator/tinystories-narrative-classification/validation.csv')

In [11]:
pd.options.display.max_rows = 500

print("Summary of training dataset:")
print(train.describe)

Summary of training dataset:
<bound method NDFrame.describe of                                                       text
0        One day, a little girl named Lily found a need...
1        Once upon a time, there was a little car named...
2        One day, a little fish named Fin was swimming ...
3        Once upon a time, in a land full of trees, the...
4        Once upon a time, there was a little girl name...
...                                                    ...
2119714  Once upon a time, in a small town, there lived...
2119715  Once upon a time, there was a little boy named...
2119716  Once upon a time, there was a big tree. Under ...
2119717  Once upon a time, there was a little girl name...
2119718  Once upon a time, there was an adorable little...

[2119719 rows x 1 columns]>


In [12]:
print("Summary of validation dataset:")
print(val.describe)

Summary of validation dataset:
<bound method NDFrame.describe of                                                     text
0      Spot. Spot saw the shiny car and said, "Wow, K...
1      Once upon a time, in a big forest, there lived...
2      Once upon a time, in a small yard, there was a...
3      Once upon a time, there was a thoughtful girl ...
4      Once upon a time, there was a kind farmer. He ...
...                                                  ...
21985  John was out in the backyard playing. He saw a...
21986  Once upon a time, there was a green vehicle. E...
21987  Once upon a time, there was a package. It was ...
21988  Once upon a time, there was a thin rhinoceros....
21989  Once there was a little girl named Sarah. She ...

[21990 rows x 1 columns]>


In [13]:
print('First story in training set:')
print(train.iloc[0]['text'])

First story in training set:
One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."

Together, they shared the needle and sewed the button on Lily's shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.


## Select sample of 10 stories for initial run 

In [14]:
print('2 more stories from training set and validation set each:')
train_sample = train.sample(10, random_state = 1)
val_sample = val.sample(2, random_state = 1)

print(train_sample.iloc[0]['text'])
print('-------------X-------------')
print(train_sample.iloc[1]['text'])
print('-------------X-------------')
print(val_sample.iloc[0]['text'])
print('-------------X-------------')
print(val_sample.iloc[1]['text'])

2 more stories from training set and validation set each:
Once upon a time, there was a big elephant named Ellie. Ellie loved to walk in the forest and eat leaves. One day, Ellie met a little bird named Billy. Billy said, "Hi, Ellie! You are so big and graceful. Can I walk with you?"

Ellie was happy to have a friend to walk with. They walked and walked until they saw a big snake. The snake was not happy and said, "Get out of my way! I am going to poke you with my sharp teeth!"

Ellie and Billy were scared and did not know what to do. Ellie said, "Please don't poke us. We just want to walk." The snake thought for a moment and said, "Okay, you can walk. But be careful, there are many dangers in the forest."

Ellie and Billy were happy to continue their walk and thanked the snake for being kind. They walked until they reached a river. Ellie said, "Let's poke our trunks in the water and have a drink." They drank the water and continued their walk, happy to have each other's company.
-----

## Tokenize Data and Create Chunks of Input-Target Pairs

Firstly, we must break the natural language text into units called **tokens** and assign a numerical Id to each called a **token ID**. The list of containing all the units in the language, be they words, sub-words, or byte-pairs, indexed by their corresponding token IDs is called the **vocabulary**. For this task, we will use a pre-existing GPT2 tokenizer to perform **Byte-Pair Encoding (BPE)**.

For the next token prediction task, a **sliding window** is defined, such that given the tokens in the window, i.e. the **context**, the next token in the sequence is considered to be the expected output. The max number of tokens that can be in the window is the **context size**. Tokenized data is grouped into these input (the tokens in the window) and target (the next expected token) pairs. These groups are called **chunks**.

In [15]:
!pip install git+https://github.com/huggingface/transformers

  Cloning https://github.com/huggingface/transformers to /tmp/pip-req-build-ydz1mkym
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /tmp/pip-req-build-ydz1mkym
  Resolved https://github.com/huggingface/transformers to commit 28034545c35acac05793695bd5c878f66e6b544e
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 20.4 MB/s eta 0:00:0000:01
  Created wheel for transformers: filename=transformers-5.19.0.dev0-py3-none-any.whl size=13927669 sha256=996044a0b81764f450fd9b4156719e2393cbccf9960697c2387d29e7c221f785
  Stored in directory: /tmp/pip-ephem-wheel-cache-1c08be0h/wheels/2b/de/48/1c5b158806820c3979e1dc7a341b68dac1231f00b1a2c9442f
Successfully built transformers
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.29.0
    Uninstalling huggingface_hub-1.2

In [16]:
import torch.nn.functional as F
import torch
from transformers import GPT2Tokenizer, BatchEncoding

In [17]:
'''
tokenizer = GPT2Tokenizer.from_pretrained('openai-community/gpt2', add_eos_token = True)
print('Vocabulary size = ', tokenizer.vocab_size)

trainTokens = tokenizer(train_sample['text'].tolist())
valTokens = tokenizer(val_sample['text'].tolist())

print(trainTokens[0])
print('---------------X--------------')
print(valTokens[0])
print('---------------X--------------')
print(trainTokens['input_ids'][0])
print('---------------X--------------')
print(len(trainTokens['input_ids']))
print('---------------X--------------')
print(tokenizer.decode(trainTokens['input_ids'][0]))
'''

"\ntokenizer = GPT2Tokenizer.from_pretrained('openai-community/gpt2', add_eos_token = True)\nprint('Vocabulary size = ', tokenizer.vocab_size)\n\ntrainTokens = tokenizer(train_sample['text'].tolist())\nvalTokens = tokenizer(val_sample['text'].tolist())\n\nprint(trainTokens[0])\nprint('---------------X--------------')\nprint(valTokens[0])\nprint('---------------X--------------')\nprint(trainTokens['input_ids'][0])\nprint('---------------X--------------')\nprint(len(trainTokens['input_ids']))\nprint('---------------X--------------')\nprint(tokenizer.decode(trainTokens['input_ids'][0]))\n"

In [57]:
class Dataset:
    def __init__(self, rawData, contextSize, split):
        self.split = split
        tokenizer = GPT2Tokenizer.from_pretrained('openai-community/gpt2', add_eos_token = True)
        tokenData = tokenizer(rawData)
        tokenIds = []
        for document in tokenData['input_ids']:
            tokenIds.extend(document)
        # Chunking tokens based on input and target
        self.inputTokens = torch.zeros((len(tokenIds) - contextSize,contextSize))
        self.targetTokens = torch.zeros((len(tokenIds) - contextSize,contextSize))
        for i in range(len(tokenIds) - contextSize):
            try:
                self.inputTokens[i,:] = torch.tensor(tokenIds[i:i+contextSize])
                self.targetTokens[i,:] = torch.tensor(tokenIds[i+1:i+contextSize+1])
            except:
                print("i = ", i)
                print("input vector = ", torch.tensor(tokenIds[i:i+contextSize]))
                print("target vector = ", torch.tensor(tokenIds[i+1:i+contextSize+1]))
                break

    def __len__(self):
        return self.inputTokens.size(dim=0)

    def __getitem__(self, idx):
        return self.inputTokens[idx,:], self.targetTokens[idx,:]

In [58]:
trainData = Dataset(train_sample['text'].tolist(), contextSize = 4, split = 'train')

In [19]:
'''
def io_pairing(data):
    inputTokens, outputTokens = [], []
    # Leave out the last token;it is never input to the model
    for document in data:    
        inputTokens.extend(document[0:len(document) - 1])
        outputTokens.extend(document[1:len(document)])

    return inputTokens, outputTokens
    '''

'\ndef io_pairing(data):\n    inputTokens, outputTokens = [], []\n    # Leave out the last token;it is never input to the model\n    for document in data:    \n        inputTokens.extend(document[0:len(document) - 1])\n        outputTokens.extend(document[1:len(document)])\n\n    return inputTokens, outputTokens\n    '

In [20]:
'''
inputTokens, outputTokens = io_pairing(trainTokens['input_ids'])
print(inputTokens[0:4])
print(outputTokens[0:4])
print(inputTokens[len(inputTokens) - 4:len(inputTokens)])
print(outputTokens[len(outputTokens) - 4:len(outputTokens)])
'''

"\ninputTokens, outputTokens = io_pairing(trainTokens['input_ids'])\nprint(inputTokens[0:4])\nprint(outputTokens[0:4])\nprint(inputTokens[len(inputTokens) - 4:len(inputTokens)])\nprint(outputTokens[len(outputTokens) - 4:len(outputTokens)])\n"

In [21]:
# inputValTokens, outputValTokens = io_pairing(valTokens['input_ids'])

### Batch Data to Prepare it for the Embedding Layer of the Model

In [22]:
# an iterable class to batch data in dataset and iterated through the batches
class DatasetLoader:
    def __init__(self, dataset, batchSize, contextSize):
        dataLen = dataset.__len__()
        assert dataLen // batchSize
        # Batching data from Dataset object
        self.numOfBatches = dataLen / batchSize
        self.currentBatch = 0
        self.dataset = dataset

    def __iter__(self):
        return self

    def __next__(self):
        inputData = torch.zeros((batchSize, contextSize))
        targetData = torch.zeros((batchSize, contextSize))
        # get data for current batch and collate it
        index = currentBatch * batchSize
        for i in range(batchSize):
            index = index + i
            inputData[index,:], outputData[index,:]= self.dataset[index] # syntactical sugar for __getitem__()
        self.currentBatch = self.currentBatch + 1
        return inputData, outputData

## Project to Embedding Space

A set of numbers, such as these **tokens**, carry no information whatsoever to a computer. To imbue these tokens with some **semantic meaning**, we project them to a higher order **embedding space**. Thus, each token becomes a vector of *n* dimensions, where *n* is the dimension of the embedding space.

These weights, first initialized randomly, will be learned by the Language Model during training.

In [24]:
# embedMatrix = torch.rand(vocabSize, emdbDim)
'''
def embedding_layer(vocabSize, embdDim, inputTokens): # perform lookup in Embedding Matrix
    return embedMatrix[inputTokens]
'''

'\ndef embedding_layer(vocabSize, embdDim, inputTokens): # perform lookup in Embedding Matrix\n    return embedMatrix[inputTokens]\n'

In [25]:
class Transformer:
    def __init__(self, attnHeads):
        self.attenHeads = attnHeads

class LangModel:
    def __init__(self, contextSize, batchSize, vocabSize):
        self.contextSize = contextSize
        self.batchSize = batchSize
        self.vocabSize = vocabSize
        self.embeddingMatrix = torch.rand(vocabSize, embdDim)

    def embedding_layer(vocabSize, embdDim, inputTokens): # perform lookup in Embedding Matrix
        return embedMatrix[inputTokens] 

In [ ]:
A = torch.tensor([[3,4,5,2], [3,1,5,1], [5,6,2,1]])
B = torch.tensor([2,1, 2])
A[B]

In [ ]:
'''
embedMatrix, embedInput, embedOutput = generate_embd(tokenizer.vocab_size, 
                                                     128, 
                                                     torch.from_numpy(inputTokens), 
                                                     torch.from_numpy(outputTokens))
embedMatrix.shape
embedInput.shape
embedOutput.shape
'''